## 1. Import Libraries and Define Bounds

The assignment gives fixed ranges for the unknown parameters. These bounds are used during optimization so the solver cannot choose invalid values.

`theta` is stored in radians because NumPy trigonometric functions use radians.

In [7]:
import math
from dataclasses import dataclass

import numpy as np
import pandas as pd
from scipy.optimize import differential_evolution,least_squares

THETA_BOUNDS=(math.radians(1e-9),math.radians(50.0))
M_BOUNDS=(-0.05,0.05)
X_BOUNDS=(0.0,100.0)
T_BOUNDS=(6.0,60.0)

@dataclass
class FitResult:
    theta:float
    M:float
    X:float
    mean_l1:float
    max_l1:float
    total_l1:float
    mean_implicit_residual:float
    max_implicit_residual:float
    t_min:float
    t_max:float


## 2. Load the Data

The CSV contains only `x` and `y` columns. Since there is no explicit `t` column and the rows are shuffled, the solution should not depend on row order.

The data is loaded as an `N x 2` NumPy array so the same vectorized calculations can be applied to every point.

In [8]:
def load_points(path):
    df=pd.read_csv(path)
    columns={name.lower().strip(): name for name in df.columns}
    if "x" not in columns or "y" not in columns:
        raise ValueError(f"Expected x and y columns, found {list(df.columns)}")
    return df[[columns["x"],columns["y"]]].to_numpy(dtype=float)

points=load_points("xy_data.csv")
points.shape

(1500, 2)

## 3. Use the Rotation Structure

The given curve can be seen as a rotated and shifted version of the simpler curve

\[
u=t, \qquad v=e^{M|t|}\sin(0.3t)
\]

The observed point `(x, y)` is produced by rotating `(u, v)` by `theta` and shifting it by `(X, 42)`.

So for a guessed `theta` and `X`, we can undo the rotation and shift:

\[
u=(x-X)\cos\theta+(y-42)\sin\theta
\]

\[
v=-(x-X)\sin\theta+(y-42)\cos\theta
\]

If the guessed parameters are correct, the recovered values must satisfy:

\[
v=e^{M|u|}\sin(0.3u)
\]

In [9]:
def invert_transform(points,theta,X):
    x=points[:,0]
    y=points[:,1]
    u=(x-X)*np.cos(theta)+(y-42.0)*np.sin(theta)
    v=-(x-X)*np.sin(theta)+(y-42.0)*np.cos(theta)
    return u, v

def oscillation(u,M):
    return np.exp(M*np.abs(u))*np.sin(0.3*u)

def reconstruct_points(points,theta,M,X):
    u,_=invert_transform(points,theta,X)
    v_hat=oscillation(u,M)
    x_hat=u*np.cos(theta)-v_hat*np.sin(theta)+X
    y_hat=42.0+u*np.sin(theta)+v_hat*np.cos(theta)
    return u,x_hat,y_hat

## 4. Define the Residuals and Score

The first residual measures whether the inverse-transformed points lie on the simpler unrotated curve.

The L1 score is computed from the difference between the original `(x, y)` points and their reconstructed positions. A small penalty is added if the recovered hidden `u` values move outside the required `6 < t < 60` range.

In [10]:
def implicit_residuals(params,points):
    theta,M,X=params
    u,v=invert_transform(points,theta,X)
    return v-oscillation(u,M)

def implicit_objective(params,points):
    residuals=implicit_residuals(params,points)
    theta,_,X=params
    u,_=invert_transform(points,theta,X)
    lower_violation=np.clip(T_BOUNDS[0]-u,0.0,None)
    upper_violation=np.clip(u-T_BOUNDS[1],0.0,None)
    penalty=np.mean(lower_violation**2+upper_violation**2)
    return float(np.mean(residuals**2)+1000.0*penalty)

def total_l1_objective(params,points):
    theta,M,X=params
    if not (THETA_BOUNDS[0]<=theta<=THETA_BOUNDS[1]
        and M_BOUNDS[0]<=M<=M_BOUNDS[1]
        and X_BOUNDS[0]<=X<=X_BOUNDS[1]):
        return 1e18

    u,x_hat,y_hat=reconstruct_points(points,theta,M,X)
    lower_violation=np.clip(T_BOUNDS[0]-u,0.0,None)
    upper_violation=np.clip(u-T_BOUNDS[1],0.0,None)
    penalty=np.sum(lower_violation**2+upper_violation**2)
    total_l1=np.sum(np.abs(x_hat-points[:,0])+np.abs(y_hat-points[:,1]))
    return float(total_l1+1e6*penalty)

## 5. Fit the Parameters

The fitting is done in two stages.

First, a global search finds a good starting region for `theta`, `M`, and `X`. Then `least_squares` refines that result using the implicit curve residual.

After that, a narrow final search directly minimizes the total L1 reconstruction error. This last step slightly improves the score because it optimizes the same style of metric used in the assignment.

In [11]:
def summarize_fit(points,theta,M,X):
    u,x_hat,y_hat=reconstruct_points(points,theta,M,X)
    abs_dx=np.abs(x_hat-points[:,0])
    abs_dy=np.abs(y_hat-points[:,1])
    l1=abs_dx+abs_dy
    implicit_abs=np.abs(implicit_residuals(np.array([theta,M,X]),points))

    return FitResult(
        theta=float(theta),
        M=float(M),
        X=float(X),
        mean_l1=float(np.mean(l1)),
        max_l1=float(np.max(l1)),
        total_l1=float(np.sum(l1)),
        mean_implicit_residual=float(np.mean(implicit_abs)),
        max_implicit_residual=float(np.max(implicit_abs)),
        t_min=float(np.min(u)),
        t_max=float(np.max(u)),
    )

def fit_parameters(points):
    bounds=[THETA_BOUNDS,M_BOUNDS,X_BOUNDS]

    coarse_result=differential_evolution(
        lambda params:implicit_objective(params,points),
        bounds=bounds,
        seed=0,
        tol=1e-9,
        polish=False,
        maxiter=100,
    )

    refined_result=least_squares(
        lambda params:implicit_residuals(params,points),
        x0=coarse_result.x,
        bounds=(
            [THETA_BOUNDS[0],M_BOUNDS[0],X_BOUNDS[0]],
            [THETA_BOUNDS[1],M_BOUNDS[1],X_BOUNDS[1]],
        ),
        xtol=1e-15,
        ftol=1e-15,
        gtol=1e-15,
        max_nfev=20000,
    )

    baseline=summarize_fit(points,*refined_result.x)
    theta0,M0,X0=refined_result.x
    local_bounds=[
        (theta0-1e-5, theta0 + 1e-5),
        (M0-1e-6,M0+1e-6),
        (X0-1e-5,X0+1e-5),
    ]

    l1_result=differential_evolution(
        lambda params:total_l1_objective(params,points),
        bounds=local_bounds,
        seed=1,
        tol=1e-12,
        polish=True,
        maxiter=200,
    )

    polished=summarize_fit(points,*l1_result.x)
    return baseline, polished

## 6. Print the Final Answer

The output shows both the geometry fit and the L1-polished fit. The polished values are the best numeric submission from this notebook.

The clean interpretable answer is also printed because the fitted values are extremely close to `theta = pi/6`, `M = 0.03`, and `X = 55`.

In [ ]:
baseline,polished=fit_parameters(points)

print("Baseline fit from geometry")
print(f"theta (rad): {baseline.theta:.15f}")
print(f"theta (deg): {math.degrees(baseline.theta):.12f}")
print(f"M          : {baseline.M:.15f}")
print(f"X          : {baseline.X:.15f}")
print(f"total L1   : {baseline.total_l1:.15f}\n")

print("Polished fit after direct L1 refinement")
print(f"theta (rad): {polished.theta:.15f}")
print(f"theta (deg): {math.degrees(polished.theta):.12f}")
print(f"M          : {polished.M:.15f}")
print(f"X          : {polished.X:.15f}")
print(f"mean L1    : {polished.mean_l1:.15e}")
print(f"max L1     : {polished.max_l1:.15e}")
print(f"total L1   : {polished.total_l1:.15f}")
print(f"mean |implicit residual|: {polished.mean_implicit_residual:.15e}")
print(f"max  |implicit residual|: {polished.max_implicit_residual:.15e}")
print(f"recovered t range      : [{polished.t_min:.12f}, {polished.t_max:.12f}]\n")

print("Clean interpretable answer")
print("theta = pi / 6, M = 0.03, X = 55")

Baseline fit from geometry
theta (rad): 0.523598303175340
theta (deg): 29.999972932158
M          : 0.029999996873060
X          : 54.999998212799468
total L1   : 0.005245138222193

Polished fit after direct L1 refinement
theta (rad): 0.523598304384940
theta (deg): 29.999973001463
M          : 0.029999997081893
X          : 54.999998339856162
mean L1    : 3.495102745915801e-06
max L1     : 2.384898179741413e-05
total L1   : 0.005242654118874
mean |implicit residual|: 2.558593111170504e-06
max  |implicit residual|: 1.745866859015344e-05
recovered t range      : [6.049405364091, 59.995170586814]

Clean interpretable answer
theta = pi / 6, M = 0.03, X = 55
